## Dữ liệu Đầu vào (X_train): Đồ thị Cảnh

Một đồ thị được định nghĩa bởi các **nút (nodes)** và các **cạnh (edges)**.

1. Đặc trưng của Nút (Node Features):
    Mỗi đối tượng trong cảnh là một nút. Nút này cần được biểu diễn bằng một vector đặc trưng.
    - **Object Label Embedding (Mã hóa nhãn đối tượng):** Máy tính không hiểu chuỗi "sofa". Chúng ta cần chuyển nó thành một vector số: Dùng một mô hình ngôn ngữ đã huấn luyện trước (như CLIP text encoder, hoặc SentenceTransformer) để lấy vector embedding cho nhãn đối tượng (ví dụ: `model.encode("sofa") -> vector 768 chiều`). Cách này giúp GAT hiểu được sự tương đồng ngữ nghĩa (ví dụ: "chair" và "armchair" sẽ có vector gần nhau).
    - **Bounding Box Dimensions (Kích thước hộp bao):** Đây là thông tin cực kỳ quan trọng. Bạn sẽ dùng vector 3 chiều `[width, height, depth]` của bounding box. Điều này giúp GAT hiểu được "độ lớn" của đối tượng để sắp xếp cho hợp lý.
    => **Vector Đặc trưng cho mỗi Nút = `concat([Label Embedding, Bounding Box Dimensions])`**.

2. Đặc trưng của Cạnh (Edge Features):
    - Đây là phần khó nhất và là mấu chốt của việc chuẩn bị dữ liệu. File .glb không có thông tin "on", "under". *Chúng ta phải suy luận (infer) ra chúng từ thông tin hình học.* Bạn sẽ tạo một bộ quy tắc (rule-based engine) để gán nhãn cho các mối quan hệ.
    - *Mã hóa One-hot:* Nếu bạn có 5 loại quan hệ (on_top_of, under, next_to, behind, in_front_of), mỗi cạnh sẽ được biểu diễn bằng một vector 5 chiều. Nếu nhiều hơn thì vector nhiều chiều hơn. 

In [25]:
# Import các thư viện cần thiết
import os
import json
import trimesh
import numpy as np
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

# Thiết lập đường dẫn
BASE_DIR = Path(r"d:\fpt\9. Fall25\AIP\preprocessed_GAT_train_data")

# Path to dataset
DATA_DIR = BASE_DIR / "3D-FRONT-TEST"

# Path to final output JSON (X_train data)
OUTPUT_JSON = BASE_DIR / "scene_objects_mapping.json"

# Các ngưỡng để xác định mối quan hệ không gian
WARDROBE_HEIGHT_THRESHOLD = 0.9

In [ ]:
def filter_objects(object_name):
    """
    Lọc bỏ các object không mong muốn: wall, floor, ceil
    Trả về True nếu object nên được giữ lại
    """
    exclude_keywords = ['wall', 'floor', 'ceil']
    object_lower = object_name.lower()
    
    for keyword in exclude_keywords:
        if keyword in object_lower:
            return False
    return True

def clean_object_label(filename_stem):
    """
    Trích xuất nhãn danh mục chính từ tên file.
    Ví dụ: "Bed_69a2550c-ec04-44d9-a621-e601330a9cb6_1" -> "Bed"
    """
    # Lấy phần đầu tiên trước dấu '_'
    category = filename_stem.split('_')[0]
    return category

def refine_label_based_on_geometry(label, object_path):
    """
    Làm giàu nhãn dựa trên thuộc tính hình học (chiều cao).
    Hiện tại áp dụng cho "Cabinet".
    """
    if label != "Cabinet":
        return label

    try:
        # Tải mesh để lấy bounding box
        mesh = trimesh.load(object_path, force='mesh')
        
        # Vì chiều cao thường là trục Y trong hệ tọa độ 3D
        height = float(mesh.bounding_box.extents[1])

        if height > WARDROBE_HEIGHT_THRESHOLD:
            # print(("Wardrobe", height, object_path))  # FOR DEBUGGING
            return "Wardrobe"
        else:
            # print(("Cabinet", height, object_path))   # FOR DEBUGGING
            return "Cabinet"
    except Exception as e:
        print(f"Warning: Không thể xử lý file {object_path}. Lỗi: {e}. Giữ nguyên nhãn '{label}'.")
        return label
    
def scan_dataset():
    """
    Quét toàn bộ dataset và tạo mapping: scene_name -> [list_of_objects]
    """
    scene_objects_mapping = defaultdict(list)
    
    # Duyệt qua tất cả các thư mục scene. Ví dụ: 3D-FRONT-HF-SAMPLE/deaa4ac0-8d13-423f-930f-78fb25825e8d
    scene_dirs = [d for d in DATA_DIR.iterdir() if d.is_dir()]

    for scene_dir in tqdm(scene_dirs, desc="Scanning scenes"):
        # Lấy tên scene. Ví dụ: deaa4ac0-8d13-423f-930f-78fb25825e8d
        scene_id = scene_dir.name
        
        # Duyệt qua các room trong scene. Ví dụ: 3D-FRONT-HF-SAMPLE/deaa4ac0-8d13-423f-930f-78fb25825e8d/Bedroom-5088
        room_dirs = [d for d in scene_dir.iterdir() if d.is_dir()]
        
        for room_dir in room_dirs:
            # Lấy tên room. Ví dụ: Bedroom-5088
            room_name = room_dir.name
            scene_key = f"{scene_id}/{room_name}"
            
            # Tìm tất cả các file .glb trong room
            glb_files = list(room_dir.glob("*.glb"))
            
            for glb_file in glb_files:
                # Lấy tên object từ tên file
                object_name = glb_file.stem  # Tên file không có extension
                
                # Lọc object
                if filter_objects(object_name):
                    scene_objects_mapping[scene_key].append({
                        "name": object_name,    # Tên file .glb
                        "path": str(glb_file),
                        "label": refine_label_based_on_geometry(clean_object_label(object_name), str(glb_file))
                    })
    
    return dict(scene_objects_mapping)

scene_mapping = scan_dataset()

In [ ]:
# Visualize check
print(len(scene_mapping.values()))

In [ ]:
# Bước 2: Trích xuất bounding box từ các file .glb
def extract_bounding_box(glb_path):
    """
    Đọc file .glb và trích xuất kích thước bounding box
    Returns: [width, height, depth] hoặc None nếu lỗi
    """
    try:
        mesh = trimesh.load(glb_path, force='mesh')
        
        # Nếu là Scene (nhiều mesh), merge lại
        if isinstance(mesh, trimesh.Scene):
            mesh = mesh.dump(concatenate=True)
        
        # Lấy bounding box
        bounds = mesh.bounds                # [[min_x, min_y, min_z], [max_x, max_y, max_z]]
        dimensions = bounds[1] - bounds[0]  # [width, height, depth]
        
        return dimensions.tolist()
    except Exception as e:
        print(f"Lỗi khi đọc {glb_path}: {e}")
        return None

def enrich_with_bounding_boxes(scene_mapping):
    """
    Thêm thông tin bounding box vào mỗi object trong scene_mapping
    Ở mỗi object sẽ có thêm trường 'bounding_box': [width, height, depth]
    """
    print("\nBắt đầu trích xuất bounding box và thêm vào scene mapping...")
    total_objects = sum(len(objects) for objects in scene_mapping.values())
    processed = 0
    
    for scene_name, objects in tqdm(scene_mapping.items(), desc="Processing scenes"):
        for obj in objects:
            bbox = extract_bounding_box(obj['path'])
            obj['bounding_box'] = bbox
            processed += 1
    
    print(f"Đã xử lý {processed}/{total_objects} objects")
    return scene_mapping

scene_mapping = enrich_with_bounding_boxes(scene_mapping)

In [ ]:
# Visualize check
first_scene = list(scene_mapping.values())[996]
if first_scene:
    print(json.dumps(first_scene[0], indent=2, ensure_ascii=False))

### Xử lý "Others" - Export ảnh để phân loại bằng VLM

Phần này sẽ:
1. Tìm tất cả các object có label "Others" 
2. Render ảnh 2D từ file .glb
3. **Export ảnh ra folder** để gửi đi phân loại bằng VLM
4. Tạo file **mapping.json** để track
5. Có function **import kết quả** từ VLM và update labels

**Output:**
- `others_images/` - Folder chứa ảnh các object Others
- `others_mapping.json` - Mapping {filename: object_info}
- Function để đọc kết quả phân loại và cập nhật lại scene_mapping

In [34]:
# Setup cho việc render và export ảnh
import pyrender
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.transform import Rotation
PYRENDER_AVAILABLE = True

# Các đường dẫn
OTHERS_IMAGES_DIR = BASE_DIR / "others_images"
OTHERS_MAPPING_JSON = BASE_DIR / "others_mapping.json"

VLM_RESULTS_JSON = BASE_DIR / "vlm_classification_results.json" # Thay đổi tên file kết quả nếu có kết quả

# Tạo thư mục nếu chưa tồn tại
OTHERS_IMAGES_DIR.mkdir(exist_ok=True)

In [35]:
def create_look_at_matrix(eye, target, up=np.array([0, 1, 0])):
    """
    Tạo CAMERA-TO-WORLD matrix cho PyRender
    """
    # Tính forward vector (hướng từ camera đến target)
    forward = target - eye
    forward_norm = np.linalg.norm(forward)
    if forward_norm < 1e-6:
        # Tránh chia cho 0 nếu eye và target trùng nhau
        return np.eye(4)
    forward = forward / forward_norm
    
    # Tính right vector
    right = np.cross(forward, up)
    right_norm = np.linalg.norm(right)
    
    # Xử lý trường hợp forward và up song song (nhìn thẳng lên hoặc xuống)
    if right_norm < 1e-6:
        # Nếu forward gần song song với Y, dùng X-axis làm "up" tạm thời
        temp_up = np.array([1, 0, 0]) if abs(np.dot(forward, up)) > 0.99 else up
        right = np.cross(forward, temp_up)
        right_norm = np.linalg.norm(right)
        
    right = right / right_norm
    
    # Tính up_corrected (vuông góc với cả right và forward để tạo hệ tọa độ trực chuẩn)
    up_corrected = np.cross(right, forward) # Chú ý: thứ tự cross product quan trọng
    
    # Ma trận CAMERA-TO-WORLD
    # Các cột của ma trận là các vector cơ sở của camera trong world space
    mat = np.eye(4)
    
    # Gán vector vào CỘT thay vì HÀNG
    mat[:3, 0] = right        # Cột X của camera trong world
    mat[:3, 1] = up_corrected # Cột Y của camera trong world  
    mat[:3, 2] = -forward     # Cột Z của camera trong world (nhìn theo -Z)
    mat[:3, 3] = eye          # Vị trí (translation) của camera trong world
    
    return mat

def create_rotation_matrix_from_direction(direction):
    """
    Tạo ma trận xoay 4x4 chỉ từ một vector hướng cho DirectionalLight.
    Ánh sáng sẽ chiếu ngược lại hướng được cung cấp.
    """
    # Hướng của ánh sáng (ngược với vector direction)
    light_dir = -np.array(direction)
    light_dir = light_dir / np.linalg.norm(light_dir)

    # Chọn một vector 'up' tùy ý, không song song với light_dir
    if np.abs(np.dot(light_dir, [0, 1, 0])) < 0.99:
        up = [0, 1, 0]
    else:
        up = [1, 0, 0]

    # Tính các vector cơ sở
    right = np.cross(up, light_dir)
    right /= np.linalg.norm(right)
    
    up_corrected = np.cross(light_dir, right)
    up_corrected /= np.linalg.norm(up_corrected)

    # Tạo ma trận xoay
    # Cột Z của ma trận là hướng của ánh sáng
    rotation_matrix = np.eye(4)
    rotation_matrix[:3, 0] = right
    rotation_matrix[:3, 1] = up_corrected
    rotation_matrix[:3, 2] = light_dir
    
    return rotation_matrix

def render_mesh_multiview(glb_path, output_size=(512, 512), num_views=4):
    """
    Render file .glb từ nhiều góc nhìn với camera pose chuẩn
    Returns: List of PIL Images hoặc [] nếu lỗi
    
    num_views: số lượng góc nhìn (mặc định 4: front, right, top, perspective)
    """
    if not PYRENDER_AVAILABLE:
        print("PyRender không khả dụng!")
        return []
    
    try:
        # Load mesh
        mesh = trimesh.load(glb_path, force='mesh')
        if isinstance(mesh, trimesh.Scene):
            mesh = mesh.dump(concatenate=True)
        
        # Fix normals nếu mesh không watertight (tránh mặt bị lật)
        if not mesh.is_watertight:
            mesh.fix_normals()
        
        # Tính center và bounding sphere radius (chính xác hơn bbox)
        bounds = mesh.bounds
        center = (bounds[0] + bounds[1]) / 2
        
        # Dùng bounding sphere radius thay vì bbox diagonal
        vertices = mesh.vertices - center  # Center về gốc tọa độ
        radius = np.max(np.linalg.norm(vertices, axis=1))
        distance = radius * 2.5  # Distance từ center đến camera
        
        # Định nghĩa các camera positions
        views_config = {
            'front': {
                'eye': center + np.array([0, 0, distance]),
                'target': center,
                'up': np.array([0, 1, 0])
            },
            'right': {
                'eye': center + np.array([distance, 0, 0]),
                'target': center,
                'up': np.array([0, 1, 0])
            },
            'top': {
                'eye': center + np.array([0, distance, 0]),
                'target': center,
                'up': np.array([0, 0, -1])  # Z-axis hướng về phía camera khi nhìn từ trên
            },
            'perspective': {
                'eye': center + np.array([distance * 0.7, distance * 0.7, distance * 0.7]),
                'target': center,
                'up': np.array([0, 1, 0])
            }
        }
        
        images = []
        renderer = pyrender.OffscreenRenderer(output_size[0], output_size[1])
        
        # Render từ mỗi góc nhìn
        for view_name, config in list(views_config.items())[:num_views]:
            # Tạo scene mới cho mỗi view
            scene = pyrender.Scene(ambient_light=[0.3, 0.3, 0.3])  # Thêm ambient light
            
            # Thêm mesh vào scene
            mesh_node = pyrender.Mesh.from_trimesh(mesh)
            scene.add(mesh_node)
            
            # Tạo camera pose bằng look_at matrix
            camera_pose = create_look_at_matrix(
                config['eye'], 
                config['target'], 
                config['up']
            )
            
            # Camera với FOV phù hợp
            camera = pyrender.PerspectiveCamera(yfov=np.pi / 3.0, aspectRatio=1.0)
            scene.add(camera, pose=camera_pose)
            
            # Thêm nhiều light sources để chiếu sáng tốt hơn
            # Hướng của các nguồn sáng (chiếu VỀ PHÍA tâm vật thể)
            camera_direction = config['eye'] - config['target']
            
            # Key light (chính) - hướng từ camera
            key_light_direction = camera_direction
            key_light_pose = create_rotation_matrix_from_direction(key_light_direction)
            key_light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=5.0)
            scene.add(key_light, pose=key_light_pose)

            # Fill light (phụ, từ bên trái camera)
            right_vec = np.cross(camera_direction, config['up'])
            fill_light_direction = camera_direction - right_vec
            fill_pose = create_rotation_matrix_from_direction(fill_light_direction)
            fill_light = pyrender.DirectionalLight(color=[0.5, 0.5, 0.5], intensity=3.0)
            scene.add(fill_light, pose=fill_pose)

            # Rim light (đèn viền, từ phía trên)
            rim_light_direction = camera_direction + config['up']
            rim_pose = create_rotation_matrix_from_direction(rim_light_direction)
            rim_light = pyrender.DirectionalLight(color=[0.6, 0.6, 0.6], intensity=2.0)
            scene.add(rim_light, pose=rim_pose)
            
            # Render
            color, _ = renderer.render(scene)
            img = Image.fromarray(color)
            images.append((view_name, img))
        
        renderer.delete()
        return images
        
    except Exception as e:
        print(f"Lỗi render {glb_path}: {e}")
        import traceback
        traceback.print_exc()
        return []

print("Setup hoàn tất.")

Setup hoàn tất.


In [ ]:
# Export ảnh multi-view của tất cả "Others" objects
def export_others_images_multiview(scene_mapping, num_views=4):
    """
    Tìm tất cả object "Others", render multi-view và lưu vào folder riêng
    
    Cấu trúc:
    others_images/
        ├── object_id_1/
        │   ├── front.png
        │   ├── right.png
        │   ├── top.png
        │   └── perspective.png
        ├── object_id_2/
        │   ├── front.png
        │   └── ...
    """
    print("\nBắt đầu export ảnh multi-view 'Others' objects...")
    
    others_mapping = {}
    others_count = 0
    exported_count = 0
    
    # Đếm số lượng "Others"
    for scene_name, objects in scene_mapping.items():
        for idx, obj in enumerate(objects):
            if "others" in obj['name'].lower():
                others_count += 1
    
    print(f"Tìm thấy {others_count} objects có label 'Others'")
    
    if others_count == 0:
        print("Không có object nào cần export!")
        return scene_mapping, others_mapping
    
    if not PYRENDER_AVAILABLE:
        print("PyRender không khả dụng. Không thể render ảnh!")
        return scene_mapping, others_mapping
    
    # Export từng object
    for scene_name, objects in tqdm(scene_mapping.items(), desc="Exporting Others multi-view"):
        for idx, obj in enumerate(objects):
            if "others" in obj['name'].lower():
                # Tạo object ID unique
                scene_id = scene_name.replace('/', '_')
                object_id = f"{scene_id}_obj{idx}"
                
                # Tạo folder cho object này
                object_folder = OTHERS_IMAGES_DIR / object_id
                object_folder.mkdir(exist_ok=True)
                
                print(f"\nExporting: {obj['name']} from {scene_name}")
                print(f"  → Folder: {object_id}")
                
                # Render multi-view
                images = render_mesh_multiview(obj['path'], num_views=num_views)
                
                if images:
                    # Lưu từng view
                    for view_name, img in images:
                        image_path = object_folder / f"{view_name}.png"
                        img.save(image_path)
                        print(f"    ✓ Saved: {view_name}.png")
                    
                    # Lưu mapping info
                    others_mapping[object_id] = {
                        'scene_name': scene_name,
                        'object_index': idx,
                        'original_name': obj['name'],
                        'glb_path': obj['path'],
                        'bounding_box': obj.get('bounding_box'),
                        'folder': str(object_folder),
                        'num_views': len(images),
                        'views': [view_name for view_name, _ in images],
                        'new_label': None  # Sẽ được cập nhật từ VLM
                    }
                    
                    exported_count += 1
                else:
                    print(f"  ✗ Failed to render")
    
    # Lưu mapping JSON
    with open(OTHERS_MAPPING_JSON, 'w', encoding='utf-8') as f:
        json.dump(others_mapping, f, indent=2, ensure_ascii=False)
    
    return scene_mapping, others_mapping

# Chạy export với 4 views (front, right, top, perspective)
scene_mapping, others_mapping = export_others_images_multiview(scene_mapping, num_views=4)

Bước tiếp theo:
1. Gửi folder './other_images' đi phân loại bằng VLM".
2. VLM sẽ xem multi-view của mỗi object trong folder con.
3. Nhận kết quả dạng JSON: {{\"object_id\": \"new_label\", ...}}"
4. Lưu kết quả vào: {VLM_RESULTS_JSON}" với định dạng:
    {
        "object_id_1": "chair",
        "object_id_2": "plant",
        ...
    }

    Mỗi object_id tương ứng với 1 folder trong `others_images/`, folder đó chứa nhiều ảnh views (front.png, right.png, top.png, perspective.png).
5. Chạy cell tiếp theo để import kết quả.

### Import kết quả phân loại từ VLM

Sau khi bạn nhận được kết quả từ VLM (file JSON), chạy cell này để update labels.

In [ ]:
def import_vlm_results(scene_mapping, vlm_results_path=VLM_RESULTS_JSON):
    """
    Đọc kết quả phân loại từ VLM và cập nhật labels vào scene_mapping
    
    Format VLM results JSON (với object_id thay vì filename):
    {
        "object_id_1": "chair",
        "object_id_2": "plant",
        ...
    }
    """
    print("\nĐang import kết quả từ VLM...")
    
    # Kiểm tra file có tồn tại không
    if not vlm_results_path.exists():
        print("Không tìm thấy file kết quả VLM.")
        return scene_mapping
    
    # Đọc VLM results
    with open(vlm_results_path, 'r', encoding='utf-8') as f:
        vlm_results = json.load(f)
    
    print(f"Đã load {len(vlm_results)} kết quả phân loại từ VLM")
    
    # Đọc mapping
    if not OTHERS_MAPPING_JSON.exists():
        print(f"Không tìm thấy mapping file: {OTHERS_MAPPING_JSON}")
        return scene_mapping
    
    with open(OTHERS_MAPPING_JSON, 'r', encoding='utf-8') as f:
        others_mapping = json.load(f)
    
    # Update labels
    updated_count = 0
    not_found_count = 0
    
    for object_id, new_label in vlm_results.items():
        if object_id not in others_mapping:
            print(f"Không tìm thấy {object_id} trong mapping")
            not_found_count += 1
            continue
        
        # Lấy thông tin object
        obj_info = others_mapping[object_id]
        scene_name = obj_info['scene_name']
        obj_idx = obj_info['object_index']
        
        # Update label trong scene_mapping
        if scene_name in scene_mapping:
            if obj_idx < len(scene_mapping[scene_name]):
                old_label = scene_mapping[scene_name][obj_idx]['label']
                scene_mapping[scene_name][obj_idx]['label'] = new_label
                scene_mapping[scene_name][obj_idx]['original_label'] = old_label
                scene_mapping[scene_name][obj_idx]['relabeled_by_vlm'] = True
                
                updated_count += 1
                print(f"✓ Updated: {object_id}: '{old_label}' -> '{new_label}'")
    
    print(f"\n{'='*70}")
    print(f"IMPORT HOÀN TẤT!")
    print(f"{'='*70}")
    print(f"Đã cập nhật: {updated_count} objects")
    print(f"Không tìm thấy: {not_found_count} objects")
    
    # Thống kê labels mới
    if updated_count > 0:
        new_labels = list(vlm_results.values())
        unique_labels = set(new_labels)
        print(f"\nCác label mới được thêm vào:")
        for label in sorted(unique_labels):
            count = new_labels.count(label)
            print(f"  - {label}: {count}")
    
    return scene_mapping

# Chạy import (sẽ báo lỗi nếu chưa có file VLM results)
scene_mapping = import_vlm_results(scene_mapping)

In [ ]:
# Bước 4: Lưu kết quả vào JSON
def save_scene_mapping(scene_mapping, output_path):
    """
    Lưu scene_mapping vào file JSON
    """
    print(f"\nĐang lưu kết quả vào {output_path}...")
    
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(scene_mapping, f, indent=2, ensure_ascii=False)
    
    print(f"Đã lưu thành công!")
    
    # Thống kê
    total_scenes = len(scene_mapping)
    total_objects = sum(len(objects) for objects in scene_mapping.values())
    
    print(f"\n{'='*60}")
    print(f"THỐNG KÊ CUỐI CÙNG")
    print(f"{'='*60}")
    print(f"Tổng số scene: {total_scenes}")
    print(f"Tổng số object: {total_objects}")
    print(f"Trung bình object/scene: {total_objects/total_scenes:.2f}")
    
    # Thống kê label
    label_counts = defaultdict(int)
    for objects in scene_mapping.values():
        for obj in objects:
            label_counts[obj['label']] += 1
    
    print(f"\nTop 10 loại object phổ biến nhất:")
    for label, count in sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:10]:
        print(f"  {label}: {count}")
    
    return True

# Lưu file JSON
save_scene_mapping(scene_mapping, OUTPUT_JSON)

### Bước 5: Tạo Node Features và Edge Features cho GAT

Bây giờ chúng ta sẽ:
1. Tạo **Label Embeddings** sử dụng SentenceTransformer
2. Kết hợp với **Bounding Box dimensions** để tạo node features hoàn chỉnh
3. Tạo **Edge features** dựa trên khoảng cách không gian giữa các object
4. Lưu thành format có thể dùng cho PyTorch Geometric

In [ ]:
# Tạo Label Embeddings
from sentence_transformers import SentenceTransformer
SENTENCE_TRANSFORMER_AVAILABLE = True

def create_label_embeddings(scene_mapping):
    """
    Tạo embeddings cho tất cả các label sử dụng SentenceTransformer
    """
    if not SENTENCE_TRANSFORMER_AVAILABLE:
        print("SentenceTransformer không khả dụng!")
        return None
    
    print("\nĐang load SentenceTransformer model...")
    model = SentenceTransformer('all-MiniLM-L6-v2')  # Model nhỏ, nhanh
    
    # Lấy tất cả unique labels
    all_labels = set()
    for objects in scene_mapping.values():
        for obj in objects:
            all_labels.add(obj['label'])
    
    all_labels = sorted(list(all_labels))
    print(f"Tìm thấy {len(all_labels)} unique labels")
    
    # Tạo embeddings
    print("Đang tạo embeddings...")
    label_embeddings = {}
    
    for label in tqdm(all_labels, desc="Creating embeddings"):
        embedding = model.encode(label)
        label_embeddings[label] = embedding.tolist()
    
    print(f"Đã tạo embeddings cho {len(label_embeddings)} labels")
    print(f"Embedding dimension: {len(next(iter(label_embeddings.values())))}")
    
    return label_embeddings

# Tạo embeddings
label_embeddings = create_label_embeddings(scene_mapping) if SENTENCE_TRANSFORMER_AVAILABLE else None

if label_embeddings:
    print("\nMẫu embedding:")
    sample_label = list(label_embeddings.keys())[0]
    print(f"Label: {sample_label}")
    print(f"Embedding (first 10 dims): {label_embeddings[sample_label][:10]}")

In [ ]:
# Tạo Node Features hoàn chỉnh
def create_node_features(scene_mapping, label_embeddings):
    """
    Tạo node features = concat([label_embedding, bounding_box])
    """
    if label_embeddings is None:
        print("Không có label embeddings!")
        return scene_mapping
    
    print("\nĐang tạo node features...")
    
    for scene_name, objects in tqdm(scene_mapping.items(), desc="Creating node features"):
        for obj in objects:
            label = obj['label']
            bbox = obj['bounding_box']
            
            if label in label_embeddings and bbox is not None:
                # Concat embedding + bounding box
                label_emb = label_embeddings[label]
                node_feature = label_emb + bbox  # List concatenation
                
                obj['node_feature'] = node_feature
                obj['node_feature_dim'] = len(node_feature)
            else:
                obj['node_feature'] = None
                print(f"Cảnh báo: Không tạo được feature cho {obj['name']}")
    
    print("Đã tạo node features!")
    return scene_mapping

# Tạo node features
if label_embeddings:
    scene_mapping = create_node_features(scene_mapping, label_embeddings)
    
    # Hiển thị mẫu
    print("\nMẫu node feature:")
    for scene_name, objects in list(scene_mapping.items())[:1]:
        for obj in objects[:1]:
            if obj.get('node_feature'):
                print(f"Object: {obj['name']}")
                print(f"Label: {obj['label']}")
                print(f"Bounding box: {obj['bounding_box']}")
                print(f"Node feature dim: {obj['node_feature_dim']}")
                print(f"Node feature (first 10): {obj['node_feature'][:10]}")

In [ ]:
# Tạo Edges dựa trên khoảng cách không gian
def compute_centroid(bbox):
    """
    Tính centroid từ bounding box
    Giả định bbox = [width, height, depth]
    """
    if bbox is None:
        return None
    return np.array(bbox) / 2.0  # Simplified centroid

def create_edges(objects, distance_threshold=3.0):
    """
    Tạo edges giữa các object dựa trên khoảng cách
    Trả về edge_index (2, num_edges) và edge_attr (num_edges, edge_feature_dim)
    """
    num_objects = len(objects)
    edge_index = []
    edge_attr = []
    
    for i in range(num_objects):
        for j in range(i + 1, num_objects):
            obj_i = objects[i]
            obj_j = objects[j]
            
            # Tính khoảng cách giữa centroids
            if obj_i['bounding_box'] and obj_j['bounding_box']:
                centroid_i = compute_centroid(obj_i['bounding_box'])
                centroid_j = compute_centroid(obj_j['bounding_box'])
                
                distance = np.linalg.norm(centroid_i - centroid_j)
                
                # Chỉ tạo edge nếu khoảng cách < threshold
                if distance < distance_threshold:
                    # Thêm edge hai chiều
                    edge_index.append([i, j])
                    edge_index.append([j, i])
                    
                    # Edge feature = [distance]
                    edge_attr.append([distance])
                    edge_attr.append([distance])
    
    return edge_index, edge_attr

def add_graph_structure(scene_mapping):
    """
    Thêm cấu trúc đồ thị (edges) vào mỗi scene
    """
    print("\nĐang tạo graph structure (edges)...")
    
    for scene_name, objects in tqdm(scene_mapping.items(), desc="Creating edges"):
        edge_index, edge_attr = create_edges(objects)
        
        # Lưu vào scene_mapping
        scene_mapping[scene_name] = {
            'objects': objects,
            'edge_index': edge_index,
            'edge_attr': edge_attr,
            'num_nodes': len(objects),
            'num_edges': len(edge_index)
        }
    
    print("Đã tạo graph structure!")
    return scene_mapping

# Tạo edges
scene_mapping = add_graph_structure(scene_mapping)

# Hiển thị mẫu
print("\nMẫu graph structure:")
first_scene_name = list(scene_mapping.keys())[0]
first_scene = scene_mapping[first_scene_name]
print(f"Scene: {first_scene_name}")
print(f"Num nodes: {first_scene['num_nodes']}")
print(f"Num edges: {first_scene['num_edges']}")
print(f"Edge index (first 5): {first_scene['edge_index'][:5]}")
print(f"Edge attr (first 5): {first_scene['edge_attr'][:5]}")

In [ ]:
# Bước 6: Lưu dữ liệu cuối cùng
def save_final_dataset(scene_mapping, output_path):
    """
    Lưu dataset cuối cùng bao gồm:
    - Node features
    - Edge index
    - Edge attributes
    - Metadata
    """
    output_file = output_path.parent / "gat_training_data.json"
    
    print(f"\nĐang lưu dataset cuối cùng vào {output_file}...")
    
    # Convert numpy arrays to lists for JSON serialization
    serializable_data = {}
    for scene_name, scene_data in scene_mapping.items():
        serializable_data[scene_name] = {
            'objects': scene_data['objects'],
            'edge_index': scene_data['edge_index'],
            'edge_attr': scene_data['edge_attr'],
            'num_nodes': scene_data['num_nodes'],
            'num_edges': scene_data['num_edges']
        }
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(serializable_data, f, indent=2, ensure_ascii=False)
    
    print(f"Đã lưu thành công!")
    
    # Thống kê cuối cùng
    print(f"\n{'='*70}")
    print(f"HOÀN THÀNH! DATASET ĐÃ SẴN SÀNG CHO GAT TRAINING")
    print(f"{'='*70}")
    print(f"Số scene: {len(serializable_data)}")
    
    total_nodes = sum(s['num_nodes'] for s in serializable_data.values())
    total_edges = sum(s['num_edges'] for s in serializable_data.values())
    
    print(f"Tổng số nodes: {total_nodes}")
    print(f"Tổng số edges: {total_edges}")
    print(f"Trung bình nodes/scene: {total_nodes/len(serializable_data):.2f}")
    print(f"Trung bình edges/scene: {total_edges/len(serializable_data):.2f}")
    
    # Kiểm tra node feature dimensions
    sample_scene = list(serializable_data.values())[0]
    if sample_scene['objects'] and sample_scene['objects'][0].get('node_feature'):
        node_dim = len(sample_scene['objects'][0]['node_feature'])
        print(f"\nNode feature dimension: {node_dim}")
        print(f"  - Label embedding: {node_dim - 3}")
        print(f"  - Bounding box: 3")
    
    print(f"\nFile output:")
    print(f"  - {output_file}")
    print(f"  - {OUTPUT_JSON}")
    
    return output_file

# Lưu dataset cuối cùng
final_output = save_final_dataset(scene_mapping, OUTPUT_JSON)

### Bước 7 (Tùy chọn): Chuyển đổi sang PyTorch Geometric Dataset

Để sử dụng với PyTorch Geometric GAT, chúng ta cần chuyển đổi dữ liệu JSON thành `torch_geometric.data.Data` objects.

In [ ]:
# Chuyển đổi sang PyTorch Geometric format
try:
    import torch
    from torch_geometric.data import Data, Dataset
    TORCH_GEOMETRIC_AVAILABLE = True
except ImportError:
    print("Cần cài đặt: pip install torch torch-geometric")
    TORCH_GEOMETRIC_AVAILABLE = False

def json_to_pyg_data(scene_data):
    """
    Chuyển đổi một scene từ JSON sang PyTorch Geometric Data object
    """
    objects = scene_data['objects']
    
    # Tạo node features matrix
    node_features = []
    for obj in objects:
        if obj.get('node_feature'):
            node_features.append(obj['node_feature'])
        else:
            # Nếu không có feature, dùng zero vector
            node_features.append([0.0] * 387)  # 384 (embedding) + 3 (bbox)
    
    x = torch.tensor(node_features, dtype=torch.float)
    
    # Tạo edge_index
    edge_index = torch.tensor(scene_data['edge_index'], dtype=torch.long).t().contiguous()
    
    # Tạo edge_attr
    edge_attr = torch.tensor(scene_data['edge_attr'], dtype=torch.float)
    
    # Tạo Data object
    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    
    return data

def create_pyg_dataset(scene_mapping):
    """
    Tạo danh sách các PyTorch Geometric Data objects
    """
    if not TORCH_GEOMETRIC_AVAILABLE:
        print("PyTorch Geometric không khả dụng!")
        return None
    
    print("\nĐang chuyển đổi sang PyTorch Geometric format...")
    
    data_list = []
    for scene_name, scene_data in tqdm(scene_mapping.items(), desc="Converting to PyG"):
        try:
            data = json_to_pyg_data(scene_data)
            data.scene_name = scene_name  # Lưu tên scene
            data_list.append(data)
        except Exception as e:
            print(f"Lỗi khi convert scene {scene_name}: {e}")
    
    print(f"Đã chuyển đổi {len(data_list)} scenes")
    return data_list

# Chuyển đổi
if TORCH_GEOMETRIC_AVAILABLE:
    pyg_dataset = create_pyg_dataset(scene_mapping)
    
    if pyg_dataset:
        print(f"\nMẫu PyG Data object:")
        print(pyg_dataset[0])
        print(f"\nNode features shape: {pyg_dataset[0].x.shape}")
        print(f"Edge index shape: {pyg_dataset[0].edge_index.shape}")
        print(f"Edge attr shape: {pyg_dataset[0].edge_attr.shape}")
        
        # Lưu dataset
        torch.save(pyg_dataset, BASE_DIR / "gat_pyg_dataset.pt")
        print(f"\nĐã lưu PyG dataset vào: {BASE_DIR / 'gat_pyg_dataset.pt'}")
else:
    print("\nBỏ qua bước chuyển đổi PyG. Bạn có thể dùng file JSON để load sau.")